<style>
.jp-Notebook, .notebook-container, .markdown-body {font-family: Arial, Helvetica, sans-serif;}
.jp-MarkdownOutput, .text_cell_render {font-size: 18px; line-height: 1.65;}
h1 {font-size: 2.25rem !important; margin-top: 0.35em !important;}
h2 {font-size: 1.65rem !important; margin-top: 1.35em !important;}
h3 {font-size: 1.25rem !important; margin-top: 1.1em !important;}
table {font-size: 0.95em;}
blockquote {border-left: 4px solid #aaa; padding-left: 1rem;}
</style>

# 05 · Simulación de Monte Carlo

<p><a href="https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/main/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/notebooks/5_MonteCarlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a></p>

> **Objetivo:** usar muestreo aleatorio repetido para convertir incertidumbre en una distribución de resultados y evaluar la estabilidad de la simulación.

<img src="../imgs/Ciclo_MC.png" alt="Ciclo Monte Carlo" width="650">

## 1. La idea

Un presupuesto tradicional entrega un número. Monte Carlo pregunta:

> **¿Qué rango de resultados podría ocurrir y con qué frecuencia?**

**modelo → entradas inciertas → muestreo → repetir → distribución de salida → decisión**

<img src="../imgs/MC_Aplicaciones.png" alt="Aplicaciones Monte Carlo" width="680">

## 2. Calentamiento: aproximar π

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng=np.random.default_rng(42)
n=20_000
x=rng.uniform(-1,1,n); y=rng.uniform(-1,1,n)
dentro=x**2+y**2<=1
pi_estimado=4*dentro.mean()
print(f"π estimado: {pi_estimado:.5f}")
print(f"error absoluto: {abs(np.pi-pi_estimado):.5f}")

## 3. Caso principal: costo anual de un proyecto

Modelaremos costos fijos, infraestructura, mantenimiento y eventos extraordinarios. Buscamos **riesgo presupuestal**, no un único número.

In [ ]:
def simular_costos(seed, n_sim=20_000):
    rng=np.random.default_rng(seed)
    fijo=45_000*12
    infraestructura=rng.normal(8_000,1_500,size=(n_sim,12))
    mantenimiento=rng.lognormal(mean=np.log(3_000),sigma=.35,size=(n_sim,12))
    ocurre=rng.random((n_sim,12))<.08
    incidente=ocurre*rng.uniform(8_000,25_000,size=(n_sim,12))
    return fijo+infraestructura.sum(axis=1)+mantenimiento.sum(axis=1)+incidente.sum(axis=1)

In [ ]:
costos=simular_costos(seed=42)
presupuesto=700_000
print(f"Media: ${costos.mean():,.0f}")
print(f"P95: ${np.percentile(costos,95):,.0f}")
print(f"P(exceder presupuesto): {(costos>presupuesto).mean():.1%}")
plt.hist(costos,bins=40)
plt.axvline(presupuesto,linestyle="--",label="Presupuesto")
plt.legend(); plt.show()

## 4. Una seed reproduce; varias seeds evalúan estabilidad

Una **seed** fija hace reproducible una secuencia pseudoaleatoria. Eso sirve para depurar y comparar.

Pero una sola seed es una sola realización del generador.

### 💡 IDEA
Usar varias seeds no vuelve mágicamente exacta una simulación. Sí ayuda a:

- detectar conclusiones dependientes de una seed particular;
- estimar la **variabilidad Monte Carlo** de media, percentiles o probabilidades;
- evaluar estabilidad;
- decidir si necesitamos más simulaciones.

La precisión mejora principalmente aumentando el número efectivo de muestras independientes y verificando convergencia.

In [ ]:
resultados=[]
for seed in range(20):
    c=simular_costos(seed=seed,n_sim=10_000)
    resultados.append({
        "seed":seed,
        "media":c.mean(),
        "p95":np.percentile(c,95),
        "riesgo_exceso":(c>presupuesto).mean()
    })
estabilidad=pd.DataFrame(resultados)
estabilidad.head()

In [ ]:
estabilidad[["media","p95","riesgo_exceso"]].agg(["mean","std","min","max"])

## 5. Múltiples seeds y Machine Learning

En ML también hay aleatoriedad en:

- train/test split;
- inicialización de pesos;
- minibatches;
- bagging;
- búsqueda de hiperparámetros.

Evaluar con una sola seed puede producir una conclusión demasiado dependiente de una partición o inicialización afortunada.

Cuando sea apropiado, repetir con varias seeds y reportar **media ± desviación estándar** ayuda a medir robustez.

### ⚠️ Matiz importante
Múltiples seeds ayudan a medir **sensibilidad a la aleatoriedad**. No corrigen por sí solas sesgo de muestreo, errores de medición, fuga de datos o sesgo estructural del modelo.

## 6. Convergencia

In [ ]:
tamanos=[500,1_000,5_000,10_000,50_000]
riesgos=[]
for n_sim in tamanos:
    c=simular_costos(seed=123,n_sim=n_sim)
    riesgos.append((c>presupuesto).mean())
pd.DataFrame({"n_sim":tamanos,"riesgo_exceso":riesgos})

## 7. Mini reto
Cambia la probabilidad de incidente. Ejecuta al menos 10 seeds y compara la variabilidad del P95 con `n_sim=1_000` y `n_sim=20_000`.

## 📚 Referencias
- [NumPy Generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [NumPy SeedSequence](https://numpy.org/doc/stable/reference/random/bit_generators/generated/numpy.random.SeedSequence.html)
- [SciPy distributions](https://docs.scipy.org/doc/scipy/reference/stats.html)

## Qué sigue
Terminamos con un puente práctico: obtener datos desde web o APIs antes de analizarlos.